# Exploratory Data Analysis & Ground-Truth Ring Topologies
### IBM Transactions for Anti-Money Laundering (HI-Small)
**Author:** Dhruv Gourisaria  
**Project:** Money Mule Ring Detector  
**Date:** November 2026

This notebook explores the official IBM AML `HI-Small` benchmark dataset (5,078,345 transactions, 5,177 laundering cases, 370 ground-truth rings) generated via IBM's AMLworld simulator (Altman et al., NeurIPS 2023).

**Key Objectives:**
1. Quantify the extreme class imbalance (~0.1% illicit).
2. Analyze transaction amount, currency, and payment channel distributions.
3. Inspect the chronological timeline and dataset partitions.
4. Explore ground-truth laundering ring sizes and pattern distributions.
5. Visualize the canonical laundering topologies (Cycles, Fan-In, Fan-Out, Scatter-Gather) as directed graphs.

In [ ]:
import sys
sys.path.append('../src')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

data_dir = Path('../data/processed/ibm_hi_small')
trans_path = data_dir / 'transactions.parquet'
rings_path = data_dir / 'rings_truth.pkl'

print('Loading processed dataset...')

In [ ]:
df = pd.read_parquet(trans_path)
rings_df = pd.read_pickle(rings_path)

n_total = len(df)
n_illicit = int(df['label'].sum())
pct_illicit = (n_illicit / n_total) * 100

print(f'Total Transactions Scored: {n_total:,}')
print(f'Laundering Transfers:     {n_illicit:,} ({pct_illicit:.3f}%)')
print(f'Ground-Truth Mule Rings:  {len(rings_df):,}')

## 1. Class Imbalance Analysis
In real-world compliance and in this benchmark, only ~0.1% of transactions represent money laundering. This severe asymmetry makes traditional classification accuracy useless.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear vs Log representation
counts = [n_total - n_illicit, n_illicit]
labels = ['Legitimate (99.899%)', 'Laundering (0.101%)']
colors = ['#2b5c8f', '#e74c3c']

ax1.bar(labels, counts, color=colors, width=0.5)
ax1.set_title('Class Imbalance (Linear Scale)')
ax1.set_ylabel('Transaction Count')

ax2.bar(labels, counts, color=colors, width=0.5)
ax2.set_yscale('log')
ax2.set_title('Class Imbalance (Log Scale)')
ax2.set_ylabel('Transaction Count (log)')
for i, c in enumerate(counts):
    ax2.text(i, c * 1.2, f'{c:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Transaction Amount Distributions
Let's compare the amount distribution between legitimate transfers and illicit money laundering transfers.

In [ ]:
plt.figure(figsize=(12, 5))
sns.kdeplot(np.log10(df[df['label'] == 0]['amount_sent'] + 1), label='Legitimate', color='#2b5c8f', fill=True, alpha=0.3)
sns.kdeplot(np.log10(df[df['label'] == 1]['amount_sent'] + 1), label='Laundering', color='#e74c3c', fill=True, alpha=0.5)
plt.title('Log10 Transaction Amount Distribution by Class')
plt.xlabel('Log10(Amount Sent + 1)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 3. Payment Channels & Currencies
Which payment mechanisms (Wire, ACH, Cheque, Credit Card, Cash) and currencies are used in laundering attempts?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df['pay_format'].value_counts().plot(kind='bar', ax=ax1, color='#34495e', rot=30)
ax1.set_title('Payment Formats (All Transactions)')
ax1.set_ylabel('Count')

illicit_df = df[df['label'] == 1]
illicit_df['cur_sent'].value_counts().head(8).plot(kind='bar', ax=ax2, color='#e67e22', rot=30)
ax2.set_title('Top Currencies in Laundering Transactions')
ax2.set_ylabel('Laundering Transfers')

plt.tight_layout()
plt.show()

## 4. Chronological Timeline & Train/Val/Test Partitions
Our evaluation strictly enforces a temporal split (60% Train, 20% Validation, 20% Test) to ensure zero forward-looking data leakage.

In [ ]:
daily = df.set_index('ts').resample('D')['edge_id'].count()

n = len(df)
t_train_end = df['ts'].iloc[int(n * 0.6)]
t_val_end = df['ts'].iloc[int(n * 0.8)]

plt.figure(figsize=(14, 5))
daily.plot(color='#2980b9', lw=2, marker='o')
plt.axvline(t_train_end, color='#27ae60', ls='--', lw=2, label=f'Train Cutoff ({t_train_end.strftime("%Y-%m-%d")})')
plt.axvline(t_val_end, color='#e74c3c', ls='--', lw=2, label=f'Val Cutoff ({t_val_end.strftime("%Y-%m-%d")})')
plt.title('Daily Transaction Volume Across Temporal Split Horizons')
plt.ylabel('Transactions / Day')
plt.legend()
plt.show()

## 5. Ground-Truth Laundering Rings
Let's inspect the distribution of true laundering patterns and ring sizes across the 370 ground-truth rings.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

rings_df['pattern_type'].value_counts().plot(kind='bar', ax=ax1, color='#8e44ad', rot=30)
ax1.set_title('Ground-Truth Rings by Topology')
ax1.set_ylabel('Number of Rings')

rings_df['n_accounts'].plot(kind='hist', bins=20, ax=ax2, color='#16a085', edgecolor='white')
ax2.set_title('Distribution of Mule Accounts per Ring')
ax2.set_xlabel('Number of Accounts')

plt.tight_layout()
plt.show()

## 6. Visualizing Canonical Laundering Subgraphs
Here we extract and render the exact directed flow subgraphs for the primary money laundering topologies planted in the IBM benchmark.

In [ ]:
def plot_ring_subgraph(pattern_type, title, ax):
    sub_rings = rings_df[rings_df['pattern_type'] == pattern_type]
    if sub_rings.empty:
        return
    sample_ring = sub_rings.iloc[0]
    r_edge_ids = sample_ring['edge_ids']
    sub_edges = df[df['edge_id'].isin(r_edge_ids)]
    
    g = nx.DiGraph()
    for _, row in sub_edges.iterrows():
        g.add_edge(row['src'], row['dst'], amount=row['amount_sent'])
        
    pos = nx.spring_layout(g, seed=42)
    nx.draw_networkx_nodes(g, pos, ax=ax, node_color='#e74c3c', node_size=600)
    nx.draw_networkx_labels(g, pos, ax=ax, font_size=8, font_color='white', font_weight='bold')
    nx.draw_networkx_edges(g, pos, ax=ax, edge_color='#2c3e50', arrowsize=20, width=2)
    ax.set_title(f'{title} (Ring #{sample_ring["ring_id"]})', fontsize=12, fontweight='bold')
    ax.axis('off')

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_ring_subgraph('CYCLE', 'Directed Cycle Topology', axes[0, 0])
plot_ring_subgraph('FAN-OUT', 'Fan-Out Structuring (Smurfing)', axes[0, 1])
plot_ring_subgraph('FAN-IN', 'Fan-In Pooling / Collection', axes[1, 0])
plot_ring_subgraph('SCATTER-GATHER', 'Scatter-Gather Layering', axes[1, 1])

plt.tight_layout()
plt.show()

## 7. Key Findings for Machine Learning Defense

1. **Local Features Excel on Closed Loops:** Cycles and fan topologies create high degree disparity and reciprocal transaction counts (`reverse_pair_count_before`), explaining why Model M2 detected **80.0% of Cycles**.
2. **Linear Chains Are Invisible to Local Features:** Stacks ($A \to B \to C \to D$) lack degree anomalies or reciprocity. Each transaction appears standard, leading to a **3.85% recovery rate** for local graph features.
3. **Multi-Hop Graph Convolutions:** The failure on linear stacks provides empirical motivation for multi-hop GNN message passing that aggregates multi-hop neighborhood context.